# 08_02 The LSTM against the RNN: does it remember what the RNN forgot?

The last notebook measured a gradient. This one measures what a gradient is for: learning. First on a task
built so that only memory can solve it, then on real news headlines, and then on the mistake the chapter's
last page described, feeding the padding to the network.

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-08-remembering-across-a-sentence", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'torch': 'torch',
           'sklearn': 'scikit-learn'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json
import os
import time
import torch
import headlines
from nlpcheck import ask, guess, reveal, check_08_02

torch.set_num_threads(4)
data = headlines.load_split()
X_train, y_train, X_test, y_test = data["X_train"], data["y_train"], data["X_test"], data["y_test"]
print(len(X_train), "training headlines,", len(X_test), "held out")

## 1. Recall

**r3.** Going back one step along an LSTM's cell state, the gradient is multiplied by what? (a) the recurrent
weights, (b) the slope of tanh, (c) the forget gate alone

**r4.** In what order does PyTorch stack an LSTM's four blocks? (a) input, forget, candidate, output,
(b) forget, input, output, candidate, (c) it keeps them in separate layers

In [ ]:
ask("r3", "")
ask("r4", "")

## 2. A task only memory can solve

Sequences of 50 made-up words; the label is the first word, one of four; the other 49 are noise. Guessing
scores 0.25. Three networks get the same data, the same size of memory (32), the same optimiser and the same
3 epochs: the RNN, a fresh LSTM, and an LSTM with its forget gate opened as in the last notebook. Predict each
one's test accuracy. Training all three takes under half a minute.

In [ ]:
guess("memory_rnn", None)         # a number between 0 and 1
guess("memory_lstm", None)
guess("memory_lstm_open", None)

In [ ]:
Xm, ym = headlines.first_word_task(3000, 50, seed=0)
Xm_test, ym_test = headlines.first_word_task(1000, 50, seed=1)
small = dict(vocab_size=headlines.TASK_VOCAB, embed_dim=8, hidden=32)
results = {"memory": {}}
for name in ("rnn", "lstm", "lstm_open"):
    torch.manual_seed(0)
    m = headlines.SequenceClassifier(cell="rnn" if name == "rnn" else "lstm", **small)
    if name == "lstm_open":
        headlines.open_forget_gate(m, 3.0)
    t = time.time()
    loss = headlines.train(m, Xm, ym, epochs=3, batch_size=32, optimizer=torch.optim.AdamW(m.parameters(), lr=5e-3))
    results["memory"][name] = headlines.evaluate(m, Xm_test, ym_test)["accuracy"]
    print(f"{name:9}  accuracy {results['memory'][name]:.3f}   loss by epoch {loss}   ({time.time() - t:.0f} s)")
for name in ("rnn", "lstm", "lstm_open"):
    reveal("memory_" + name, round(results["memory"][name], 2))

The RNN scores about 0.24 and the fresh LSTM about 0.25: both are guessing. Their loss never leaves 1.386,
which is `ln 4`, the loss of spreading belief evenly over four answers. The LSTM with its forget gate open scores
about 0.76 after three epochs, and its loss is still falling.

So the LSTM is not memory for free. The fresh one has the same cell state, but its forget gates start near 0.5,
and 49 halvings leave almost nothing of the first word, so training has almost no signal to open them with. Start the gates
open and the first word's signal reaches the loss from the first batch; the network then learns to keep the one
thing that matters. This is the whole chapter in three numbers: the RNN cannot carry the word, the LSTM can,
and it has to learn to.

## 3. Real headlines: the RNN against the LSTM

Now the real data. To keep each run short, both networks train on the first 20,000 training headlines for 2
epochs, packed, and are scored on all the held-out ones. About a minute for both.

In [ ]:
n = 20000
results["headlines"] = {}
def run(name, model):
    torch.manual_seed(0)
    t = time.time()
    headlines.train(model, X_train[:n], y_train[:n], epochs=2)
    results["headlines"][name] = headlines.evaluate(model, X_test, y_test)
    print(f"{name:12} {results['headlines'][name]}   ({time.time() - t:.0f} s)")

torch.manual_seed(0); run("rnn", headlines.SequenceClassifier(cell="rnn"))
torch.manual_seed(0); run("lstm", headlines.SequenceClassifier(cell="lstm"))

About 0.80 for the RNN and 0.84 for the LSTM. The gap is real and much smaller than on the memory task, for a
reason you can see in the data: a headline is nine words on average, and its category is mostly carried by one
or two of them ("ebola", "oscar", "fed"), so even the RNN only has to remember a few steps.

## 4. Feed it the padding

Every headline is padded on the right to 20 ids. `pack=False` makes the network read all 20, so a nine-word
headline's final state comes after eleven steps of zeros, which is what the book's code would have done had it
padded on the right. Predict the RNN's accuracy when it reads the padding. Training takes under a minute.

In [ ]:
guess("padded_rnn", None)   # a number between 0 and 1

In [ ]:
torch.manual_seed(0); run("rnn_padded", headlines.SequenceClassifier(cell="rnn", pack=False))
reveal("padded_rnn", round(results["headlines"]["rnn_padded"]["accuracy"], 2))

About 0.26: guessing, with a macro-F1 near 0.18, because it ends up predicting almost one category for
everything. Nothing about the headlines changed. What changed is that every headline's final state now comes
after about eleven steps of zeros, so the error signal has to cross eleven steps of the RNN before it reaches a
single real word, and it arrives as nothing. This is the vanishing gradient on real data, and it is exactly the
mistake the chapter's last page described.

## 5. Your turn: the LSTM, reading the padding

Build the same LSTM as in section 3 but with `pack=False`, and run it through `run("lstm_padded", ...)`. About
under a minute.

In [ ]:
torch.manual_seed(0)
lstm_padded = None   # YOUR CODE HERE: a headlines.SequenceClassifier LSTM that reads the padding
if lstm_padded is not None:
    run("lstm_padded", lstm_padded)

In [ ]:
os.makedirs("out", exist_ok=True)
json.dump(results, open("out/08_02_results.json", "w"), indent=1)
check_08_02()

The LSTM reading the padding scores about 0.78, against the RNN's 0.26 and its own 0.84 when packed. It learned
to hold its memory through eleven steps of zeros, because a forget gate near 1 is exactly what "do nothing with
this input" needs. It still loses six points to packing, and it spends more than twice the steps: packing is not
a trick for weak networks, it is the right way to feed any of them.

## 6. Exit ticket

Explain it back, in the cell below, in two sentences of your own: why did eleven steps of zeros ruin the RNN and
not the LSTM?

**x3.** The first-word task's loss sat near 1.386 and then fell. What does 1.386 mean? (a) the network has
learned half the task, (b) the learning rate is too high, (c) it is ln 4, the loss of guessing evenly among
four classes, so falling below it means the network has started to carry the first word

In [ ]:
my_explanation = ""
ask("x3", "")